In [1]:
# Importar dependências necessárias
from dotenv import load_dotenv  # Para carregar variáveis de ambiente do .env
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient  # SDK da Typesafe para modelo Jev

from pprint import pprint  # Para exibição formatada de dados

# Introdução ao Modelo Jev

Jev é um modelo de linguagem especializado em **análise estruturada de texto** desenvolvido pela Typesafe. 
Diferente de modelos de linguagem genéricos, Jev é otimizado para responder perguntas com formatos predefinidos (Choice, Score, Noul).

## Características principais:
- **Respostas estruturadas**: Retorna dados estruturados em vez de texto livre
- **Análise de confiança**: Fornece confidence scores e probabilidades
- **Eficiência de tokens**: Otimizado para usar poucos tokens (economia de custos)
- **Multilíngue**: Funciona bem em português e outras línguas

In [2]:
# Carregar variáveis de ambiente do arquivo .env
# Isso precisa ser feito antes de inicializar o client para que TYPESAFE_API_KEY seja acessível
load_dotenv()

True

In [3]:
# Inicializar cliente da API Typesafe
# Usa TYPESAFE_API_KEY da variável de ambiente carregada anteriormente
client = TypeSafeClient()

## Teste simples de uso do Jev

## Tipos de Respostas Estruturadas

O Jev suporta três tipos de respostas:

### 1. **Choice** - Escolha entre opções
- Retorna uma única opção escolhida
- Fornece confiança e probabilidades para cada opção
- Ideal para: categorização, roteamento, classificação

### 2. **Score** - Avaliação em escala
- Retorna um valor numérico dentro de uma escala definida
- Fornece confiança e distribuição de probabilidades
- Ideal para: priorização, avaliação de severidade, scoring

### 3. **Noul** - Probabilidade sim/não
- Retorna valor entre 0 e 1 (0=não, 1=sim)
- Representa confiança em uma afirmação
- Ideal para: detecção de intenções, verificação de características

In [4]:
# Exemplo prático: análise de ticket de suporte com Jev
# O modelo responderá a perguntas estruturadas sobre o texto do ticket

# Ticket de exemplo: problema de cobrança duplicada
ticket = """
Minha assinatura foi cobrada duas vezes este mês.
Preciso do estorno o mais rápido possível,
caso contrário vou cancelar minha conta.
"""

# Definir perguntas estruturadas para o modelo analisar
# Três tipos de respostas disponíveis:
# - Choice: escolher entre opções (com probabilidades)
# - Score: avaliar em escala numérica (0-2 neste caso)
# - Noul: probabilidade sim/não (0-1)
questions = {
    "departamento": Choice(
        instructions="Qual departamento deveria tratar esta solicitação?",
        criteria={
            "financeiro": "Pagamentos, cobranças e estornos",
            "tecnico": "Bugs e problemas técnicos",
            "vendas": "Novos contratos e preços",
        },
    ),

    "urgencia": Score(
        instructions="Qual o nível de urgência da solicitação?",
        criteria=[
            "Baixa",
            "Média",
            "Alta",
        ],
    ),

    "risco_cancelamento": Noul(
        instructions="O cliente demonstra intenção de cancelar o serviço?",
    ),
}

# Enviar para análise do modelo Jev (system_one é o endpoint padrão)
response = client.system_one(
    state=ticket,
    questions=questions,
)

# Exibir resposta completa
print(response)

model='jev-1.13.0' usage=Usage(input_tokens=441, output_tokens=83) answers={'departamento': ChoiceAnswer(type='choice', choice='financeiro', confidence=1.0, probabilities={'tecnico': 0.0, 'vendas': 0.0, 'financeiro': 1.0}), 'urgencia': ScoreAnswer(type='score', score=2.0, confidence=1.0, legend={0: 'Baixa', 1: 'Média', 2: 'Alta'}, probabilities={0: 0.0, 1: 0.0, 2: 1.0}), 'risco_cancelamento': NoulAnswer(type='noul', noul=0.91)}


### Resultados da Análise

O modelo analisou o ticket e retornou:
1. **Departamento**: financeiro (100% confiança)
2. **Urgência**: Alta (score 2, 100% confiança)  
3. **Risco de cancelamento**: 91% de probabilidade que cliente cancelará se não for resolvido

In [5]:
# Inspecionar modelo usado (Jev versão 1.13.0)
print(response.model)

jev-1.13.0


In [6]:
# Ver token usage (input/output) para monitorar custos da API
print(response.usage)

input_tokens=441 output_tokens=83


In [7]:
# Acessar respostas estruturadas do modelo para cada pergunta
print(response.answers)

{'departamento': ChoiceAnswer(type='choice', choice='financeiro', confidence=1.0, probabilities={'tecnico': 0.0, 'vendas': 0.0, 'financeiro': 1.0}), 'urgencia': ScoreAnswer(type='score', score=2.0, confidence=1.0, legend={0: 'Baixa', 1: 'Média', 2: 'Alta'}, probabilities={0: 0.0, 1: 0.0, 2: 1.0}), 'risco_cancelamento': NoulAnswer(type='noul', noul=0.91)}


In [8]:
# Inspecionar resposta do tipo Choice
# Mostra: escolha feita, confiança (0-1) e probabilidades para cada opção
print(response.answers['departamento'])

type='choice' choice='financeiro' confidence=1.0 probabilities={'tecnico': 0.0, 'vendas': 0.0, 'financeiro': 1.0}


In [9]:
# Inspecionar resposta do tipo Score
# Mostra: valor score (neste caso 2=Alta), confiança e probabilidades para cada nível
print(response.answers['urgencia'])

type='score' score=2.0 confidence=1.0 legend={0: 'Baixa', 1: 'Média', 2: 'Alta'} probabilities={0: 0.0, 1: 0.0, 2: 1.0}


## Probabilidade de sim

In [10]:
# Inspecionar resposta do tipo Noul (probabilidade sim/não)
# Valor 0.91 = 91% de probabilidade que o cliente demonstra intenção de cancelar
print(response.answers['risco_cancelamento'])

type='noul' noul=0.91
